# NeuroRoute — Colab training

Pure-RL PCB router: **many nets, 6–8 layers, learned differential pairs, learned length tuning**. No LLM. No KiCad push-and-shove solver doing the hard part.

- Architecture and rationale: `neuroroute/DESIGN.md`
- What is already verified and what is not: `neuroroute/README.md`

**This notebook does NOT need the compiled `pcbworld_pns_bridge`.** That build (`notebooks/00_setup.ipynb`) is ~40 minutes of KiCad-from-source compilation and belongs to the *old* PNS thread. NeuroRoute's environment is pure PyTorch, and its KiCad validation uses `kicad-cli` from the ordinary apt package.

## Run order — do not skip ahead

| § | Cell | Why it matters |
|---|---|---|
| 1 | Setup | clone, GPU check, Drive for checkpoints |
| 2 | **Preflight** | one command, seven checks. **If it fails, stop and report.** |
| 3 | Baselines | the numbers training has to beat |
| 4 | Training | stage 0 → 1 → 3, live output |
| 5 | Artifacts | logs, curves, rendered failures — what to report back |

Set **Runtime → Change runtime type → GPU** before §4.

## 1. Setup

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/Klutzhehe/Routerv3.git'
ROOT = '/content/Routerv3'

if not os.path.isdir(ROOT):
    subprocess.run(['git', 'clone', REPO, ROOT], check=True)
else:
    subprocess.run(['git', '-C', ROOT, 'pull', '--ff-only'], check=False)

os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY -- set Runtime > GPU')
print('cpus', os.cpu_count())

In [ ]:
# Checkpoints to Drive. Colab VMs are reclaimed without warning and
# docs/RL_PLAN.md lists this as non-negotiable. `--resume` picks up from here.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/neuroroute_checkpoints'
except Exception as exc:
    print('no Drive (checkpoints will die with the VM):', exc)
    CKPT = '/content/neuroroute_checkpoints'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)

In [ ]:
# KiCad, for the sim-to-real DRC gate. Ordinary apt package, ~2 min.
# NOT the source build -- that is a different thread entirely.
!apt-get -qq update && apt-get -qq install -y kicad > /dev/null 2>&1
!kicad-cli version

## 2. Preflight — the gate

One command, seven independent checks: environment, imports, lattice geometry vs brute force, environment invariants, refine phase, **real KiCad DRC**, and a real training step (forward *and* backward) on this device.

**If any check fails, stop here and report the full output.** Every number a training run produces downstream is meaningless if the geometry it trains against is not legal — that is what check 6 exists to prove.

Takes ~3–6 minutes.

In [ ]:
!python -m neuroroute.scripts.preflight --out /content/preflight_out

## 3. Baselines — the numbers to beat

`greedy` walks straight down the geodesic gradient; `layer_hop` adds a via when another layer is closer. Because direction index 0 *is* the gradient direction, an untrained near-zero-init policy behaves like `greedy` — training starts **at** the baseline, not below it.

Measured locally for reference (yours will differ with GPU/seed): 1 net/2L greedy 75.0% vs layer_hop 87.5%; 20 nets/2L 28.1% vs 42.5%; 60 nets/8L 16.3% vs 24.6%.

In [ ]:
import time, warnings; warnings.filterwarnings('ignore')
import torch
from neuroroute.env.baselines import greedy_safe_action, detour_action, layer_hop_action
from neuroroute.env.route_env import EnvConfig, NeuroRouteEnv
from neuroroute.world.engine import WorldConfig
from neuroroute.world.generator import GeneratorConfig
from neuroroute.world.spec import BoardSpec, LayerStack

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = list(range(900000, 900008))   # held out; never trained on

def bench(nets, layers, size, batch, heads, steps, label):
    spec = BoardSpec(height_cells=size, width_cells=size, layers=LayerStack(num_layers=layers))
    print(f'\n{label}')
    for name, fn in (('greedy', greedy_safe_action), ('detour', detour_action), ('layer_hop', layer_hop_action)):
        env = NeuroRouteEnv(EnvConfig(spec=spec,
            world=WorldConfig(batch_size=batch, max_heads=heads, max_nets=max(64, nets+8),
                              max_steps_per_net=steps, device=DEV),
            generator=GeneratorConfig(num_nets=nets, num_components=max(3, nets//4)),
            max_episode_steps=steps*5))
        obs = env.reset(SEEDS[:batch]); t0 = time.perf_counter()
        for _ in range(steps*5):
            obs, r, d, i = env.step(fn(obs))
            if bool(d.all()): break
        st = env.world.board_stats()
        print(f'  {name:>10}: completion {float(env.world.completion().mean()):6.1%}  '
              f'vias {float(st["vias"].float().mean()):5.1f}  {time.perf_counter()-t0:5.1f}s')

bench(1,  2,  64, 8, 2, 64, '1 net, empty, 2 layers   (stage 0)')
bench(20, 2,  64, 8, 4, 64, '20 nets, 2 layers        (stage 1)')
bench(60, 8, 128, 8, 8, 96, '60 nets, 8 layers        (stage 3)')

In [ ]:
# Throughput. This is the number the whole design exists for: every previous
# thread in this repo ran ONE board per process on 2 vCPUs.
spec = BoardSpec(height_cells=128, width_cells=128, layers=LayerStack(num_layers=8))
env = NeuroRouteEnv(EnvConfig(spec=spec,
    world=WorldConfig(batch_size=16, max_heads=8, max_nets=128, max_steps_per_net=96, device=DEV),
    generator=GeneratorConfig(num_nets=60, num_components=8), max_episode_steps=512))
obs = env.reset()
for _ in range(5): obs, *_ = env.step(layer_hop_action(obs))       # warm up
if DEV == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter(); N = 50
for _ in range(N): obs, *_ = env.step(layer_hop_action(obs))
if DEV == 'cuda': torch.cuda.synchronize()
dt = time.perf_counter() - t0
print(f'{N*16*8/dt:,.0f} routing decisions/sec on {DEV}  (B=16, K=8, 8 layers, 128x128)')
if DEV == 'cuda':
    print(f'peak GPU memory {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

## 4. Training

Every run writes to its checkpoint dir, continuously and flushed:

| file | what |
|---|---|
| `train_log.jsonl` | one JSON per update, `fsync`-ed — survives a killed VM |
| `console.log` | full console mirror |
| `run_config.json` | environment + config, incl. git commit and dirty flag |
| `curves.png` | completion / reward / losses / health / forecaster |
| `renders/` | worst board per eval + contact sheet, failed nets dashed red |
| `latest.pt` | atomic checkpoint (`--resume` picks it up) |
| `crash_report.txt` | **only if it died** — env, traceback, tensor state, last 10 updates |

Two lines to watch, and they are not the reward:

- **`completion` vs the baselines in the EVAL block.** This repo has a measured case of a policy scoring *worse* reward while routing *more* nets, so a reward curve alone can move the wrong way and look like progress.
- **`FORECAST GATE`.** If the learned occupancy forecast never beats the straight-line demand baseline, the forecaster has learned nothing worth carrying and the latent-rollout stage in `DESIGN.md` §5 **does not start**. That gate exists so this cannot become negative result #5 by momentum — four previous lookahead efforts here ran well past the point the evidence had answered the question.

`[WARN]` / `[FATAL]` lines are health checks (NaN, entropy collapse, exploding value loss, clip fraction, rejected-action rate). A `[FATAL]` stops the run *before* corruption reaches a checkpoint.

### Stage 0 — plumbing

One net on an empty board. Should climb to ~100%. **Anything well below that is broken plumbing, not a hard problem** — do not move on from a partial result. (Note the best *non-learned* baseline here is 87.5%; the rest is cross-layer nets needing a via.)

~5 minutes.

In [ ]:
!python -m neuroroute.training.run \
    --stage 0 --device cuda --batch 16 --heads 4 --width 32 \
    --rollout 32 --updates 300 --eval-every 25 --render-every 50 --drc-every 100 \
    --checkpoint-dir {CKPT}/stage0 --resume

### Stage 1 — congestion

20 nets, two layers. The bar is the `greedy` number from §3. Gate is 75%.

In [ ]:
!python -m neuroroute.training.run \
    --stage 1 --device cuda --batch 16 --heads 8 --width 48 \
    --rollout 32 --updates 1500 --eval-every 50 --render-every 100 --drc-every 200 \
    --checkpoint-dir {CKPT}/stage1 --resume

### Stage 3 — eight layers

The first stage where vias are the main lever, and the capability KiCad's PNS router is **0-for-32** on after three sessions (`docs/RL_PLAN.md`, Gate A). There is no prior number in this repo to compare against — only the `layer_hop` baseline.

If you hit VRAM limits: drop `--batch` first, then `--width`, then `--heads`. Keep `--store-device cpu`.

In [ ]:
!python -m neuroroute.training.run \
    --stage 3 --device cuda --batch 12 --heads 8 --width 64 \
    --rollout 32 --updates 4000 --eval-every 100 --render-every 200 --drc-every 400 \
    --store-device cpu \
    --checkpoint-dir {CKPT}/stage3 --resume

## 5. Artifacts — what to report back

In [ ]:
import json, glob, os
from pathlib import Path

RUN = f'{CKPT}/stage0'   # <-- set to whichever stage you just ran

print('=== files ===')
for f in sorted(Path(RUN).rglob('*')):
    if f.is_file():
        print(f'  {f.stat().st_size:>10,}  {f.relative_to(RUN)}')

crash = Path(RUN) / 'crash_report.txt'
if crash.exists():
    print('\n=== CRASH REPORT (report this verbatim) ===')
    print(crash.read_text())
else:
    rows = [json.loads(l) for l in open(f'{RUN}/train_log.jsonl')]
    print(f'\n=== {len(rows)} updates logged ===')
    print('first:', json.dumps(rows[0])[:400])
    print('last :', json.dumps(rows[-1])[:400])
    evals = [r for r in rows if 'policy/completion' in r]
    if evals:
        print(f'\n=== {len(evals)} evals ===')
        print(f"{'upd':>6} {'policy':>8} {'greedy':>8} {'detour':>8} {'layer_hop':>10} {'rej':>7} {'fcast>base':>11}")
        for e in evals:
            print(f"{e['update']:>6} {e['policy/completion']:>8.1%} {e['greedy/completion']:>8.1%} "
                  f"{e['detour/completion']:>8.1%} {e['layer_hop/completion']:>10.1%} "
                  f"{e['policy/rejected_action_rate']:>7.2%} {str(bool(e.get('beats_baseline', 0))):>11}")
        best = max(evals, key=lambda e: e['policy/completion'])
        print(f"\nbest policy completion {best['policy/completion']:.1%} at update {best['update']}")

In [ ]:
# Learning curves. Completion is plotted alongside reward deliberately -- see
# the note in section 4 about the two disagreeing.
from IPython.display import Image, display
p = f'{RUN}/curves.png'
display(Image(p)) if os.path.exists(p) else print('no curves yet (need one eval)')

In [ ]:
# Rendered failures. Dashed red = a net that did NOT route, drawn pad-to-pad
# over the copper that is in its way. This is the debugging artifact that a
# reward curve cannot give you.
for p in sorted(glob.glob(f'{RUN}/renders/*.png'))[-4:]:
    print(p); display(Image(p))

In [ ]:
# Export a trained board to .kicad_pcb and DRC it with real KiCad.
# Download the file and open it in KiCad to look at the copper directly.
import torch
from neuroroute.env.observation import FIELD_CHANNELS, head_feature_dim, net_feature_dim
from neuroroute.models.policy import NeuroRoutePolicy
from neuroroute.eval.kicad_export import export_board
from neuroroute.scripts.validate_kicad import run_drc, summarise, classify
from neuroroute.training.curriculum import default_curriculum, stage_env_config
from pathlib import Path

STAGE = 0
ck = torch.load(f'{RUN}/latest.pt', map_location=DEV, weights_only=False)
stage = default_curriculum(layers_max=ck['args']['layers'], size=ck['args']['size'])[STAGE]
wc = WorldConfig(batch_size=4, max_heads=ck['args']['heads'],
                 max_nets=max(64, stage.generator.num_nets+8), device=DEV)
env = NeuroRouteEnv(stage_env_config(stage, wc, EnvConfig(seed=0)))
L = stage.board.num_layers
pol = NeuroRoutePolicy(FIELD_CHANNELS, head_feature_dim(L), net_feature_dim(),
                       num_layers=L, num_via_classes=stage.board.rules.num_via_classes,
                       num_width_classes=stage.board.rules.num_width_classes,
                       width=ck['args']['width']).to(DEV)
pol.load_state_dict(ck['policy']); pol.eval()

obs = env.reset(list(range(900000, 900004)))
with torch.no_grad():
    for _ in range(env.cfg.max_episode_steps):
        obs, r, d, i = env.step(pol.act(obs, deterministic=True).actions)
        if bool(d.all()): break
print('completion', float(env.world.completion().mean()))

out = Path('/content/export'); out.mkdir(exist_ok=True)
print(export_board(env.world, 0, out/'trained.kicad_pcb'))
ok, rep = run_drc(out/'trained.kicad_pcb', out/'trained.drc.json')
print('DRC readable:', ok)
if ok:
    for g, c in classify(summarise(rep)).items():
        if c: print(f'  {g}: {c}')